## Simple MDP

#### Import packages

In [ ]:
import numpy as np
import mdptoolbox
import mdptoolbox.example
import gymnasium as gym

#### Import MDP

In [ ]:
class SimpleLinearMDP:
    def __init__(
        self,
        n,
        mu_theta,
        Sigma_theta,
        alpha,
        beta,
        gamma,
        sigma_s,
        sigma_r,
        mu_s0,
        sigma_s0,
        seed=None,
    ):
        self.n = n
        self.rng = np.random.default_rng(seed)

        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.sigma_s = sigma_s
        self.sigma_r = sigma_r
        self.mu_s0 = mu_s0
        self.sigma_s0 = sigma_s0

        # theta_i ~ N(mu_theta, Sigma_theta)
        self.theta = self.rng.multivariate_normal(mu_theta, Sigma_theta, size=n)

        # current state for each participant
        self.s = None

    def reset(self):
        # s_{i,1} ~ N(mu_s0, sigma_s0^2)
        self.s = self.rng.normal(self.mu_s0, self.sigma_s0, size=self.n)
        return self.s.copy()

    def step(self, a):
        """
        a: array of shape (n,) with entries 0 or 1
        returns: reward, next_state
        """
        a = np.asarray(a)

        # x_{i,t} = (1, s_{i,t}, a_{i,t}, s_{i,t} a_{i,t})
        X = np.column_stack([
            np.ones(self.n),
            self.s,
            a,
            self.s * a
        ])

        # r_{i,t} = theta_i^T x_{i,t} + eta_{i,t}
        reward_mean = np.sum(self.theta * X, axis=1)
        reward = reward_mean + self.rng.normal(0, self.sigma_r, size=self.n)

        # s_{i,t+1} = alpha + beta s_{i,t} + gamma a_{i,t} + epsilon_{i,t}
        next_state_mean = self.alpha + self.beta * self.s + self.gamma * a
        next_state = next_state_mean + self.rng.normal(0, self.sigma_s, size=self.n)

        self.s = next_state
        return reward, next_state.copy()

In [10]:
P

array([[[0.5, 0.5],
        [0.8, 0.2]],

       [[0. , 1. ],
        [0.1, 0.9]]])

In [11]:
R

array([[ 5, 10],
       [-1,  2]])